# Missing Data Is Not Empty Space
## Statistical analysis of an irregular blood-pressure tracker

This executable notebook treats the **observation process** as part of the statistical problem. The public repository contains only a privacy-safe, day-indexed aggregate snapshot; private source rows are never committed. This is a statistical case study, not medical advice.

The analysis now also asks whether conclusions change after adjusting for **ambient temperature in Fiães at the actual blood-pressure measurement times**.

In [ ]:
from pathlib import Path
import json
import analysis as primary
import observation_process_sensitivity as observation_sensitivity
import gap_aware_trend_decomposition as gap_aware
import day_influence_sensitivity as influence
import episode_observation_sensitivity as episode_observation
import episode_time_form_sensitivity as time_form
import temporal_dependence_diagnostics as temporal

root = Path.cwd()
data_path = root / 'data/analysis_snapshot.csv'
if not data_path.exists():
    raise FileNotFoundError('Run from Blood_Pressure_Missingness/')
records = primary.load_snapshot(data_path)
observed = primary.observed_records(records)
print(f'{len(records)} calendar days; {len(observed)} observed; {len(records)-len(observed)} missing')

## 1. Observation design first

The current live snapshot has **26 observed days across 65 calendar days** and a 32-day uninterrupted gap. Sampling intensity is highly uneven. This does not mean that 171 readings are treated as exchangeable i.i.d. observations; the estimand remains at the observed-calendar-day level.

In [ ]:
global_fit = primary.linear_trend(records, 'mean_systolic_mmHg')
print(f"Global systolic trend: {global_fit['slope_per_30_days']:.2f} mmHg/30d; 95% CI [{global_fit['ci95_low_per_30_days']:.2f}, {global_fit['ci95_high_per_30_days']:.2f}]")

## 2. Gap-aware and observation-process sensitivity

The long gap prevents a smooth-trajectory interpretation. The analysis separates the global trend, the gap-defined episode contrast, sampling-intensity sensitivity, and single-day influence.

In [ ]:
gap_results = gap_aware.analyze_gap_aware_trend(records)
centered = gap_results['episode_centered_model']
print(f"Post-minus-pre contrast: {centered['post_minus_pre_mean_difference_mmHg']:.2f} mmHg")
obs_results = observation_sensitivity.fit_sensitivity_models(observation_sensitivity.load_observed_days(data_path))
print('Observation-process specifications:', len(obs_results['trend_estimates']))
influence_results = influence.summarize_influence(records)
print('Leave-one-day-out summary available:', bool(influence_results['leave_one_day_out_summary']))

## 3. Functional-form sensitivity

On the current snapshot the episode contrast remains below zero under **all five tested within-episode time forms**. The most flexible specification is treated as a stress test rather than a preferred trajectory.

In [ ]:
time_results = time_form.fit_episode_time_form_sensitivity(records)
episode_obs = episode_observation.fit_episode_observation_sensitivity(records)
print('Within-episode time-form specifications:', len(time_results['estimates']))
print('Episode observation specifications:', len(episode_obs['estimates']))

## 4. Temporal dependence respects calendar distance

Residual pairs are compared by **exact calendar-day lag within the same gap-defined episode**. The analysis therefore preserves actual calendar spacing, and **row-order adjacency should not be treated as a daily time lag**.

In [ ]:
temporal_results = temporal.diagnose_temporal_dependence(records)
spacing = temporal_results['observed_order_spacing']
print('Observed-row spacing counts:', spacing['calendar_gap_days_counts'])

## 5. Time-matched ambient temperature in Fiães

For each private reading, the refresh workflow retrieves hourly **2 m air temperature for Fiães, Santa Maria da Feira, Portugal** and matches it to the actual local measurement time. For observed day $d$,

$$T_d=\frac{1}{n_d}\sum_{i=1}^{n_d}T(t_{di}),$$

so $T_d$ is the mean ambient temperature at the times readings were actually taken, not the daily meteorological mean. The primary exploratory adjustment is

$$Y_d=\beta_0+\beta_1 d+\beta_2(T_d-\bar T)+\varepsilon_d,$$

with equal weight per observed day and HC3 robust covariance. The coefficient is an association, not a causal effect. Calendar dates, row-level measurement times, and the matched temperature sequence remain private.

In [ ]:
temperature_path = root / 'figures/temperature_covariate.json'
if temperature_path.exists():
    temperature_result = json.loads(temperature_path.read_text(encoding='utf-8'))
    systolic = temperature_result['model']['metrics']['mean_systolic_mmHg']
    print(f"Temperature-adjusted systolic trend: {systolic['day_slope_per_30_days']:.2f} mmHg/30d")
    print(f"Temperature association: {systolic['temperature_coefficient_per_c']:.2f} mmHg/°C")
else:
    print('Run the secret-backed refresh workflow to generate temperature_covariate.json.')

### Temperature robustness

The robustness layer challenges three choices: nearest-hour matching versus linear interpolation, equal-day weighting versus reading-count weighting, and a linear temperature term versus a quadratic centered-temperature term. The purpose is specification stability rather than selecting the most favorable estimate.

In [ ]:
sensitivity_path = root / 'figures/temperature_covariate_sensitivity.json'
if sensitivity_path.exists():
    sensitivity_result = json.loads(sensitivity_path.read_text(encoding='utf-8'))
    systolic_specs = sensitivity_result['metrics']['mean_systolic_mmHg']
    for name in (
        'primary_nearest_hour_equal_day_linear',
        'linear_interpolation_equal_day_linear',
        'nearest_hour_reading_weighted_linear',
    ):
        estimate = systolic_specs[name]
        print(
            f"{name}: trend={estimate['day_slope_per_30_days']:.2f} mmHg/30d; "
            f"temperature={estimate['temperature_coefficient_per_c']:.2f} mmHg/°C"
        )
    quadratic = systolic_specs['nearest_hour_equal_day_quadratic']
    print(
        'nearest_hour_equal_day_quadratic: '
        f"trend={quadratic['day_slope_per_30_days']:.2f} mmHg/30d; "
        f"linear temperature at mean={quadratic['linear_temperature_coefficient_per_c_at_mean']:.2f} mmHg/°C; "
        f"quadratic={quadratic['quadratic_temperature_coefficient_per_c2']:.3f} mmHg/°C²"
    )
    print('Direction stability:', systolic_specs['robustness'])
else:
    print('Run the secret-backed refresh workflow to generate temperature_covariate_sensitivity.json.')

## 6. Statistical conclusion

The strongest conclusion is **not** that blood pressure followed a smooth downward trajectory over 65 days. The data are dominated by a long unobserved interval and an uneven measurement process. The temperature analysis adds measurement context but does not remove the identification problem: season, time of day, activity, sleep, medication timing, and the observation process may all be related to both ambient temperature and blood pressure.

The broader lesson is unchanged but stronger: **missingness and measurement context are part of the statistical process.**